# Netflix Movies & TV Shows — Data Cleaning

This notebook loads the **Netflix Titles** dataset and cleans it with Pandas.

The dataset has missing values, mixed-type columns, and even a data-entry error.
The goal is **not** to blindly `dropna()` — it's to *understand why* each value is
missing or wrong, then make a deliberate decision per column.

**Plan**
1. Load & inspect (`.info()`, `.describe()`, `.head()`)
2. Audit missing values
3. Handle missing values column by column (with reasoning)
4. Fix a data-entry error (duration leaked into the `rating` column)
5. Split the mixed-type `duration` column into a number + unit
6. Parse `date_added` into a real datetime
7. Final check & export to a clean CSV

## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 2. Load the dataset

In [2]:
df = pd.read_csv("netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## 3. Inspect before changing anything

Always *read* the data before transforming it. `.info()` shows dtypes and non-null
counts, `.describe()` summarises columns, and `.head()` shows real examples.

In [3]:
df.shape

(8807, 12)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [5]:
# include='all' so we also summarise the text (object) columns
df.describe(include="all")

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
count,8807,8807,8807,6173,7982,7976,8797,8807.000000,8803,8804,8807,8807
unique,8807,2,8807,4528,7692,748,1767,NaN,17,220,514,8775
top,s1,Movie,Dick Johnson Is Dead,Rajiv Chilaka,David Attenborough,United States,"January 1, 2020",NaN,TV-MA,1 Season,"Dramas, International Movies","Paranormal activity at a lush, abandoned prope..."
freq,1,6131,1,19,19,2818,109,NaN,3207,1793,362,4
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014.180198,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.819312,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1925.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000,NaN,NaN,NaN,NaN


## 4. Audit missing values

Count nulls per column and express them as a percentage so we can judge how
serious each gap is.

In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "missing_%": missing_pct}).sort_values("missing", ascending=False)

,missing,missing_%
director,2634,29.91
country,831,9.44
cast,825,9.37
date_added,10,0.11
rating,4,0.05
duration,3,0.03
show_id,0,0.00
type,0,0.00
title,0,0.00
release_year,0,0.00


## 5. Handle missing values — column by column

Different columns need different decisions:

| Column | Missing | Decision | Why |
|---|---|---|---|
| `director` | ~30% | Fill with `"Unknown"` | Too many to drop; a missing director is itself meaningful (e.g. some TV shows). |
| `cast` | ~9% | Fill with `"Unknown"` | Same reasoning — dropping would lose ~800 valid rows. |
| `country` | ~9% | Fill with `"Unknown"` | We can't reliably infer it; "Unknown" keeps the row usable. |
| `date_added` | 10 | Parse to datetime; keep `NaT` | Only 10 rows — too few to distort anything. |
| `rating` | 4 (+errors) | Fix errors first, then fill | Some values are actually durations (a data-entry shift). |
| `duration` | 3 | Recovered from the `rating` fix | The 3 missing durations are the rows where duration leaked into `rating`. |


### `director`, `cast`, `country` → fill with `"Unknown"`

In [7]:
for col in ["director", "cast", "country"]:
    df[col] = df[col].fillna("Unknown")

df[["director", "cast", "country"]].isnull().sum()

director    0
cast        0
country     0
dtype: int64

## 6. Fix the data-entry error: durations stuck in `rating`

Notice that `rating` contains values like `"74 min"`. Ratings should be things
like `TV-MA` or `PG-13` — these are clearly **durations** that landed in the wrong
column, and the matching `duration` cell is empty for those rows.

So we **move** those values into `duration`, then blank out the bad `rating`.

In [8]:
# rows where rating is actually a duration (contains "min")
mask = df["rating"].astype(str).str.contains("min", na=False)
print("Rows with a duration stuck in 'rating':", mask.sum())
df.loc[mask, ["title", "rating", "duration"]]

Rows with a duration stuck in 'rating': 3


,title,rating,duration
5541,Louis C.K. 2017,74 min,NaN
5794,Louis C.K.: Hilarious,84 min,NaN
5813,Louis C.K.: Live at the Comedy Store,66 min,NaN


In [9]:
# move the value into duration, then clear the wrong rating
df.loc[mask, "duration"] = df.loc[mask, "rating"]
df.loc[mask, "rating"] = np.nan

print("duration nulls now:", df["duration"].isnull().sum())
print("rating nulls now:", df["rating"].isnull().sum())

duration nulls now: 0
rating nulls now: 7


### Remaining missing `rating` → fill with `"Unknown"`
We won't guess a content rating, so we mark it explicitly.

In [10]:
df["rating"] = df["rating"].fillna("Unknown")
df["rating"].isnull().sum()

0

## 7. Fix the mixed-type `duration` column

`duration` mixes two things: movies use minutes (`"90 min"`) and shows use seasons
(`"2 Seasons"`). A single column with two meanings is hard to analyse, so we split it:

- `duration_int` — the number (90, 2, …)
- `duration_unit` — `"Minutes"` or `"Seasons"`

In [11]:
# pull the number out of the string -> integer
df["duration_int"] = df["duration"].str.extract(r"(\d+)").astype("Int64")

# label the unit based on type (Movie = Minutes, TV Show = Seasons)
df["duration_unit"] = np.where(df["type"] == "Movie", "Minutes", "Seasons")

df[["type", "duration", "duration_int", "duration_unit"]].head()

,type,duration,duration_int,duration_unit
0,Movie,90 min,90,Minutes
1,TV Show,2 Seasons,2,Seasons
2,TV Show,1 Season,1,Seasons
3,TV Show,1 Season,1,Seasons
4,TV Show,2 Seasons,2,Seasons


## 8. Parse `date_added` into a real datetime

It's currently text like `"September 25, 2021"` (sometimes with stray spaces).
We strip the whitespace and convert to `datetime`, then pull out the year and month
so they're ready for time-based analysis. The 10 unparseable/missing rows become `NaT`.

In [12]:
df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), format="%B %d, %Y", errors="coerce")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month

df[["title", "date_added", "year_added", "month_added"]].head()

,title,date_added,year_added,month_added
0,Dick Johnson Is Dead,2021-09-25,2021.0,9.0
1,Blood & Water,2021-09-24,2021.0,9.0
2,Ganglands,2021-09-24,2021.0,9.0
3,Jailbirds New Orleans,2021-09-24,2021.0,9.0
4,Kota Factory,2021-09-24,2021.0,9.0


## 9. Final check
Confirm dtypes are sensible and no unexpected nulls remain.

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   show_id        8807 non-null   object        
 1   type           8807 non-null   object        
 2   title          8807 non-null   object        
 3   director       8807 non-null   object        
 4   cast           8807 non-null   object        
 5   country        8807 non-null   object        
 6   date_added     8797 non-null   datetime64[ns]
 7   release_year   8807 non-null   int64         
 8   rating         8807 non-null   object        
 9   duration       8807 non-null   object        
 10  listed_in      8807 non-null   object        
 11  description    8807 non-null   object        
 12  duration_int   8807 non-null   Int64         
 13  duration_unit  8807 non-null   object        
 14  year_added     8797 non-null   float64       
 15  month_added    8797 n

In [14]:
pd.DataFrame({"missing": df.isnull().sum()}).query("missing > 0")

,missing
date_added,10
year_added,10
month_added,10


**Remaining nulls are expected:** only `date_added` / `year_added` / `month_added`
for the 10 rows that never had a date. Every other column is now clean.

## 10. Export the cleaned dataset

In [15]:
df.to_csv("netflix_titles_cleaned.csv", index=False)
print("Saved netflix_titles_cleaned.csv with", df.shape[0], "rows and", df.shape[1], "columns")

Saved netflix_titles_cleaned.csv with 8807 rows and 16 columns


## Summary of decisions

- **Read first, then clean** — used `.info()`/`.describe()`/`.head()` before touching anything.
- **`director`, `cast`, `country`** → filled with `"Unknown"` instead of dropping ~30%/9%/9% of rows.
- **Data-entry error** → moved leaked durations out of `rating` into `duration`.
- **`duration`** → split the mixed column into `duration_int` + `duration_unit`.
- **`date_added`** → parsed to real datetime and derived `year_added` / `month_added`.
- **Kept 10 genuinely-missing dates as `NaT`** rather than inventing values.